In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# Rotating magnetic field in 3-phase winding

# sampling data
samples = 100
thetas = np.linspace(0, 2*np.pi, samples, endpoint=False)

# plot initialisation
plt.style.use("dark_background")
fig, ax = plt.subplots(
    1, 2, figsize=(14, 6), 
    gridspec_kw={"width_ratios": [3,2]}
)
fig.subplots_adjust(
    left=0.07, right=0.98,
    bottom=0.05, top=0.9,
    wspace=0.3, hspace=0
)
fig.suptitle("Rotating Magnetic Field in 3-Phase Winding", fontsize=30)

# signal plot formatting
ax[0].set_xlim(0, 360)
ax[0].xaxis.set_major_locator(ticker.MultipleLocator(60))
ax[0].set_aspect(110)
ax[0].set_title("3-Phase Current Signals", fontsize=20)
ax[0].set_xlabel(
    r"Mechanical angle $\theta_\text{mech}$ ($\degree$)", fontsize=15
)
ax[0].set_ylabel(r"Current magnitude $I$ ($A$)", fontsize=15)

# rotating magnetic field plot formatting
ax[1].set_aspect("equal")
ax[1].set_title(r"Magnetic Flux Densities $\mathbf{B}$", fontsize=20)
ax[1].set_xlabel(r"$B_x$ ($T$)", fontsize=15)
ax[1].set_ylabel(r"$B_y$ ($T$)", fontsize=15)


# static plot element angles
plot_angles = np.linspace(0, 2*np.pi, 200, endpoint=True)

# 3-phase signals
offsets = [n * 2*np.pi / 3 for n in range(3)]
colours = ["red", "yellow", "blue"]
for offset, colour in zip(offsets, colours):
    ax[0].plot(
        np.degrees(plot_angles), 
        np.sin(plot_angles + offset), 
        linewidth=2, color=colour
    )

# magnetic field magnitude
ax[1].plot(
    1.5*np.cos(plot_angles), 1.5*np.sin(plot_angles), 
    linestyle="--", color="white", alpha=0.5
)

# current mechanical angle slider
slider, = ax[0].plot([], [], linestyle="--", color="white", alpha=0.5)

# individual phase flux densities
args = [0, 0, 0, 0]
kwargs = {"scale": 1, "scale_units": "xy", "angles": "xy"}
phase1 = ax[1].quiver(*args, **kwargs, color="red")
phase2 = ax[1].quiver(*args, **kwargs, color="yellow")
phase3 = ax[1].quiver(*args, **kwargs, color="blue")
phases = [phase1, phase2, phase3]

# total flux density
total_b = ax[1].quiver(*args, **kwargs, color="white")

def update(frame):

    # current angle slide
    theta = thetas[frame]

    # initial flux density vector start point
    tail = [0, 0]

    # update slider
    slider.set_data([np.degrees(theta), np.degrees(theta)], [-10, 10])

    for i in range(3):

        # signal magntiude
        signal = np.sin(thetas - offsets[i])

        # current flux density vector
        tip = [
            -signal[frame] * np.cos(offsets[i]), 
            -signal[frame] * np.sin(offsets[i])
        ]

        # update flux density vector
        phases[i].set_offsets([tail])
        phases[i].set_UVC(*tip)

        # update next vector startin point
        tail[0] += tip[0]
        tail[1] += tip[1]

    # update total flux density vector
    total_b.set_offsets([0, 0])
    total_b.set_UVC(*tail)

    return slider, phase1, phase2, phase3, bfield

# animate rotation
anim = FuncAnimation(fig, update, frames=samples, blit=True)
anim.save("rotating-magnetic-field.gif", writer=PillowWriter(fps=20))
plt.close(fig)
HTML(anim.to_jshtml())